# VPTS Iceberg tutorial (PySpark)
Run in EMR Studio after attaching the tutorial cluster. Every query prints its elapsed time.

In [ ]:
from time import perf_counter

def timed_sql(label, query, rows=100):
    started = perf_counter()
    result = spark.sql(query)
    result.show(rows, truncate=False)
    print(f'TIMING | {label} | {perf_counter() - started:.2f} seconds')
    return result

## Discover the Iceberg tables

In [ ]:
timed_sql('List VPTS tables', 'SHOW TABLES IN glue_catalog.vpts')

## Basic VPTS profile queries

In [ ]:
timed_sql('Filtered profile sample', """
SELECT radar, datetime, height, dens, ff, dd
FROM glue_catalog.vpts.data
WHERE year = 2024 AND month = 5 AND rad = 'KBUF'
ORDER BY datetime, height
LIMIT 100
""")

In [ ]:
timed_sql('Daily profile summary', """
SELECT radar, CAST(datetime AS DATE) AS date,
       ROUND(AVG(dens), 2) AS mean_density,
       ROUND(AVG(CASE WHEN ISNAN(ff) THEN NULL ELSE ff END), 2) AS mean_speed,
       COUNT(*) AS profiles
FROM glue_catalog.vpts.data
WHERE year = 2024 AND month = 5 AND rad IN ('KBUF', 'KTYX')
GROUP BY radar, CAST(datetime AS DATE)
ORDER BY date, radar
LIMIT 100
""")

## VPI query

In [ ]:
timed_sql('Weekly VPI summary', """
SELECT radar, year, week,
       ROUND(AVG(CASE WHEN ISNAN(mtr) THEN NULL ELSE mtr END), 2) AS mean_mtr,
       ROUND(AVG(CASE WHEN ISNAN(vid) THEN NULL ELSE vid END), 2) AS mean_vid,
       ROUND(AVG(CASE WHEN ISNAN(ff) THEN NULL ELSE ff END), 2) AS mean_speed,
       COUNT(*) AS observations
FROM glue_catalog.vpts.vpi
WHERE year = 2024 AND radar IN ('KBUF', 'KTYX')
GROUP BY radar, year, week
ORDER BY year, week, radar
LIMIT 100
""")

## Five migration and data-quality questions

In [ ]:
timed_sql('Busiest migration weeks', """
SELECT radar, week,
       ROUND(AVG(CASE WHEN ISNAN(mtr) THEN NULL ELSE mtr END), 2) AS mean_mtr,
       ROUND(MAX(CASE WHEN ISNAN(mtr) THEN NULL ELSE mtr END), 2) AS peak_mtr
FROM glue_catalog.vpts.vpi
WHERE year = 2024 AND week BETWEEN 12 AND 22 AND radar IN ('KBUF', 'KTYX')
GROUP BY radar, week
ORDER BY peak_mtr DESC
LIMIT 20
""")

In [ ]:
timed_sql('Density-weighted flight height', """
SELECT radar, CAST(datetime AS DATE) AS date,
       ROUND(SUM(height * dens) / SUM(dens), 0) AS density_weighted_height_m,
       ROUND(MAX(dens), 2) AS peak_density
FROM glue_catalog.vpts.data
WHERE year = 2024 AND month = 5 AND rad IN ('KBUF', 'KTYX')
  AND dens > 0 AND NOT ISNAN(dens)
GROUP BY radar, CAST(datetime AS DATE)
ORDER BY peak_density DESC
LIMIT 30
""")

In [ ]:
timed_sql('Peak migration hours', """
SELECT radar, HOUR(datetime) AS utc_hour, ROUND(AVG(dens), 2) AS mean_density
FROM glue_catalog.vpts.data
WHERE year = 2024 AND month = 5 AND rad IN ('KBUF', 'KTYX')
  AND dens IS NOT NULL AND NOT ISNAN(dens)
GROUP BY radar, HOUR(datetime)
ORDER BY mean_density DESC
""")

In [ ]:
timed_sql('Mean movement vectors', """
SELECT radar,
       ROUND(AVG(CASE WHEN ISNAN(u) THEN NULL ELSE u END), 2) AS mean_u,
       ROUND(AVG(CASE WHEN ISNAN(v) THEN NULL ELSE v END), 2) AS mean_v,
       ROUND(AVG(CASE WHEN ISNAN(ff) THEN NULL ELSE ff END), 2) AS mean_speed
FROM glue_catalog.vpts.data
WHERE year = 2024 AND month = 5 AND rad IN ('KBUF', 'KTYX')
GROUP BY radar
""")

In [ ]:
timed_sql('Radar data gaps', """
SELECT radar, COUNT(*) AS profiles,
       ROUND(100 * AVG(CASE WHEN gap THEN 1.0 ELSE 0.0 END), 2) AS gap_percent
FROM glue_catalog.vpts.data
WHERE year = 2024 AND month = 5 AND rad IN ('KBUF', 'KTYX')
GROUP BY radar
ORDER BY gap_percent DESC
""")

## Full-archive scan
Which 100 m mean-flight-height bands have the highest VID? This intentionally omits year/week/radar filters.

In [ ]:
timed_sql('Archive-wide VID by flight-height band', """
SELECT FLOOR(height_mean / 100) * 100 AS height_band_m,
       COUNT(*) AS observations, COUNT(DISTINCT radar) AS radars,
       ROUND(AVG(vid), 2) AS mean_vid,
       ROUND(PERCENTILE_APPROX(vid, 0.5), 2) AS median_vid,
       ROUND(PERCENTILE_APPROX(vid, 0.95), 2) AS p95_vid
FROM glue_catalog.vpts.vpi
WHERE height_mean IS NOT NULL AND NOT ISNAN(height_mean)
  AND vid IS NOT NULL AND NOT ISNAN(vid)
GROUP BY FLOOR(height_mean / 100) * 100
HAVING COUNT(*) >= 1000
ORDER BY mean_vid DESC
""")